# Danka Ch 1 — Linear Algebra Sanity Demo

**Source:** Danka, *Mathematics of Machine Learning*, Chapter 1 (§1.01–1.04, §1.06–1.07)

This notebook verifies core linear algebra identities hands-on. Each demo maps directly to a section in Danka's text and to concepts that appear constantly in deep learning.

| Demo | Danka Section | ML Connection |
|------|--------------|---------------|
| 1. Dot product & norms | §1.03–1.04 | Loss functions, regularisation, cosine similarity |
| 2. Matrix–vector multiplication | §1.06 | Forward pass through a linear layer |
| 3. Linearity identities | §1.07 | Why neural network layers compose |
| 4. Inner product → norm → metric | §1.03–1.04 | The hierarchy behind distance-based algorithms |

In [1]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)
PASS, FAIL = "\u2713", "\u2717"
all_checks: list[bool] = []

---
## Demo 1: Dot Product and Norms (Danka §1.03–1.04)

### Theory

The **dot product** (standard inner product) in $\mathbb{R}^n$:

$$\langle \mathbf{x}, \mathbf{y} \rangle = \mathbf{x}^\top \mathbf{y} = \sum_{i=1}^{n} x_i y_i$$

A **norm** $\|\cdot\|: V \to [0, \infty)$ measures vector magnitude. Three axioms:
1. **Positive definiteness:** $\|\mathbf{x}\| = 0 \iff \mathbf{x} = \mathbf{0}$
2. **Homogeneity:** $\|c\mathbf{x}\| = |c|\|\mathbf{x}\|$
3. **Triangle inequality:** $\|\mathbf{x} + \mathbf{y}\| \leq \|\mathbf{x}\| + \|\mathbf{y}\|$

The **$p$-norm family:**

$$\|\mathbf{x}\|_p = \left(\sum_{i=1}^{n} |x_i|^p\right)^{1/p}$$

| $p$ | Name | Formula | ML use |
|-----|------|---------|--------|
| 1 | Manhattan / L1 | $\sum |x_i|$ | Lasso (sparsity) |
| 2 | Euclidean / L2 | $\sqrt{\sum x_i^2}$ | Ridge, MSE, distances |
| $\infty$ | Max / Chebyshev | $\max |x_i|$ | Adversarial robustness |

**Cosine similarity** measures the angle between vectors:

$$\cos\theta = \frac{\langle \mathbf{x}, \mathbf{y} \rangle}{\|\mathbf{x}\| \cdot \|\mathbf{y}\|}$$

In [2]:
x = np.array([3.0, 4.0])
y = np.array([1.0, 2.0])
print(f"x = {x}")
print(f"y = {y}")

x = [3. 4.]
y = [1. 2.]


### 1a. Dot product — two equivalent computations

In [4]:
dot_manual = np.sum(x * y)       # component-wise multiply then sum
dot_numpy  = np.dot(x, y)        # numpy built-in

print(f"Manual:  Σ xᵢyᵢ = {dot_manual}")
print(f"np.dot:  x·y    = {dot_numpy}")

ok = np.allclose(dot_manual, dot_numpy)
all_checks.append(ok)
print(f"{PASS if ok else FAIL} Manual == np.dot")

Manual:  Σ xᵢyᵢ = 11.0
np.dot:  x·y    = 11.0
✓ Manual == np.dot


### 1b. Norms — L1, L2, L∞

For $\mathbf{x} = (3, 4)$:
- $L^2$ comes from **Pythagoras**: $\sqrt{3^2 + 4^2} = 5$ (the 3-4-5 triangle)
- $L^1$ is the **grid walk**: $|3| + |4| = 7$
- $L^\infty$ is the **furthest axis**: $\max(3, 4) = 4$

In [6]:
l1   = np.linalg.norm(x, ord=1)
l2   = np.linalg.norm(x, ord=2)
linf = np.linalg.norm(x, ord=np.inf)

print(f"||x||₁  = {l1}   (Manhattan — expected 7)")
print(f"||x||₂  = {l2}   (Euclidean — expected 5, Pythagoras)")
print(f"||x||∞  = {linf}   (Chebyshev — expected 4)")

for name, val, exp in [("L1", l1, 7), ("L2", l2, 5), ("L∞", linf, 4)]:
    ok = np.allclose(val, exp)
    all_checks.append(ok)
    print(f"{PASS if ok else FAIL} {name} == {exp}")

||x||₁  = 7.0   (Manhattan — expected 7)
||x||₂  = 5.0   (Euclidean — expected 5, Pythagoras)
||x||∞  = 4.0   (Chebyshev — expected 4)
✓ L1 == 7
✓ L2 == 5
✓ L∞ == 4


### 1c. Homogeneity axiom: $\|c\mathbf{x}\| = |c|\|\mathbf{x}\|$

Scaling the vector by $c$ scales the norm by $|c|$. This is one of the three norm axioms.

In [9]:
c = -2.5
lhs = np.linalg.norm(c * x) # left-hand side: ||c·x||
rhs = abs(c) * np.linalg.norm(x) # right-hand side: |c|·||x||

print(f"||{c}·x||    = {lhs:.4f}")
print(f"|{c}|·||x||  = {rhs:.4f}")

ok = np.allclose(lhs, rhs)
all_checks.append(ok)
print(f"{PASS if ok else FAIL} Homogeneity holds")

||-2.5·x||    = 12.5000
|-2.5|·||x||  = 12.5000
✓ Homogeneity holds


### 1d. Cosine similarity and orthogonality

Cosine similarity is the standard similarity metric in embeddings, retrieval, and attention.

Orthogonal vectors ($\mathbf{x} \perp \mathbf{y}$) have $\cos\theta = 0$.

In [10]:
cos_sim = np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))
angle_deg = np.degrees(np.arccos(np.clip(cos_sim, -1, 1)))
print(f"cos(θ) = {cos_sim:.4f}")
print(f"θ      = {angle_deg:.2f}°")
print()

# Orthogonal check: standard basis vectors
e1, e2 = np.array([1.0, 0.0]), np.array([0.0, 1.0])
cos_ortho = np.dot(e1, e2) / (np.linalg.norm(e1) * np.linalg.norm(e2))
ok = np.allclose(cos_ortho, 0.0)
all_checks.append(ok)
print(f"cos(e₁, e₂) = {cos_ortho}")
print(f"{PASS if ok else FAIL} Orthogonal → cos(θ) = 0")

cos(θ) = 0.9839
θ      = 10.30°

cos(e₁, e₂) = 0.0
✓ Orthogonal → cos(θ) = 0


---
## Demo 2: Matrix–Vector Multiplication (Danka §1.06)

### Theory

Given $A \in \mathbb{R}^{m \times n}$ and $\mathbf{x} \in \mathbb{R}^n$, the product $A\mathbf{x}$ can be interpreted two ways:

1. **Row interpretation:** $(A\mathbf{x})_i = \text{row}_i \cdot \mathbf{x}$ (dot product of each row with $\mathbf{x}$)
2. **Column interpretation:** $A\mathbf{x} = x_1 \cdot \text{col}_1 + x_2 \cdot \text{col}_2 + \cdots$ (weighted sum of columns)

**Danka's key insight:** Column $j$ of $A$ is the image of basis vector $\mathbf{e}_j$. The columns tell you exactly what the linear transformation *does*.

**ML connection:** A linear layer computes $\hat{\mathbf{y}} = X W + \mathbf{b}$. Understanding matrix multiplication = understanding what your network does to data at each layer.

In [11]:
A = np.array([[2.0, -1.0], [1.0, 3.0]])
x = np.array([3.0, 2.0])
result = A @ x

print(f"A = [[{A[0,0]}, {A[0,1]}],")
print(f"     [{A[1,0]}, {A[1,1]}]]")
print(f"x = {x}")
print(f"Ax = {result}")

A = [[2.0, -1.0],
     [1.0, 3.0]]
x = [3. 2.]
Ax = [4. 9.]


### 2a. Row interpretation

In [12]:
row_result = np.array([np.dot(A[0], x), np.dot(A[1], x)])
print(f"row₀ · x = {A[0]} · {x} = {np.dot(A[0], x)}")
print(f"row₁ · x = {A[1]} · {x} = {np.dot(A[1], x)}")

ok = np.allclose(result, row_result)
all_checks.append(ok)
print(f"{PASS if ok else FAIL} Ax == [row_i · x]")

row₀ · x = [ 2. -1.] · [3. 2.] = 4.0
row₁ · x = [1. 3.] · [3. 2.] = 9.0
✓ Ax == [row_i · x]


### 2b. Column interpretation (Danka's key insight)

In [13]:
col_result = x[0] * A[:, 0] + x[1] * A[:, 1]
print(f"Ax = x₀·col₀ + x₁·col₁")
print(f"   = {x[0]}·{A[:, 0]} + {x[1]}·{A[:, 1]}")
print(f"   = {x[0]*A[:, 0]} + {x[1]*A[:, 1]}")
print(f"   = {col_result}")

ok = np.allclose(result, col_result)
all_checks.append(ok)
print(f"{PASS if ok else FAIL} Ax == Σ xⱼ·colⱼ(A)")

Ax = x₀·col₀ + x₁·col₁
   = 3.0·[2. 1.] + 2.0·[-1.  3.]
   = [6. 3.] + [-2.  6.]
   = [4. 9.]
✓ Ax == Σ xⱼ·colⱼ(A)


### 2c. Columns = images of basis vectors

In [14]:
e1, e2 = np.array([1.0, 0.0]), np.array([0.0, 1.0])
print(f"A @ e₁ = {A @ e1}  (col₀ = {A[:, 0]})")
print(f"A @ e₂ = {A @ e2}  (col₁ = {A[:, 1]})")

for i, (ei, name) in enumerate([(e1, 'e₁'), (e2, 'e₂')]):
    ok = np.allclose(A @ ei, A[:, i])
    all_checks.append(ok)
    print(f"{PASS if ok else FAIL} A @ {name} == col_{i}(A)")

A @ e₁ = [2. 1.]  (col₀ = [2. 1.])
A @ e₂ = [-1.  3.]  (col₁ = [-1.  3.])
✓ A @ e₁ == col_0(A)
✓ A @ e₂ == col_1(A)


---
## Demo 3: Linearity — $A(a\mathbf{x} + b\mathbf{y}) = aA\mathbf{x} + bA\mathbf{y}$ (Danka §1.07)

### Theory

A function $f: U \to V$ is a **linear transformation** if:

$$f(a\mathbf{x} + b\mathbf{y}) = af(\mathbf{x}) + bf(\mathbf{y})$$

This single equation encodes both:
- **Additivity:** $f(\mathbf{x} + \mathbf{y}) = f(\mathbf{x}) + f(\mathbf{y})$
- **Homogeneity:** $f(a\mathbf{x}) = af(\mathbf{x})$

Consequences:
- $f(\mathbf{0}) = \mathbf{0}$ (linear maps fix the origin)
- $(AB)\mathbf{z} = A(B\mathbf{z})$ (matrix multiplication = function composition)

**ML connection:** Each layer in a neural network is a linear transformation followed by a nonlinearity. Linearity lets us compose layers (backpropagation = chain rule through composed linear maps + pointwise nonlinearities).

In [15]:
rng = np.random.default_rng(42)
m, n = 3, 4
A = rng.standard_normal((m, n))
x = rng.standard_normal(n)
y = rng.standard_normal(n)
a, b = 2.5, -1.3

print(f"A shape: {A.shape} (maps ℝ⁴ → ℝ³)")
print(f"a = {a}, b = {b}")

A shape: (3, 4) (maps ℝ⁴ → ℝ³)
a = 2.5, b = -1.3


In [16]:
# Additivity
ok = np.allclose(A @ (x + y), A @ x + A @ y)
all_checks.append(ok)
print(f"{PASS if ok else FAIL} A(x + y) == Ax + Ay")

# Homogeneity
ok = np.allclose(A @ (a * x), a * (A @ x))
all_checks.append(ok)
print(f"{PASS if ok else FAIL} A(ax) == a(Ax)")

# Full linearity
ok = np.allclose(A @ (a * x + b * y), a * (A @ x) + b * (A @ y))
all_checks.append(ok)
print(f"{PASS if ok else FAIL} A(ax + by) == aAx + bAy")

# Origin preservation
ok = np.allclose(A @ np.zeros(n), np.zeros(m))
all_checks.append(ok)
print(f"{PASS if ok else FAIL} A(0) == 0")

# Composition
B = rng.standard_normal((n, 5))
z = rng.standard_normal(5)
ok = np.allclose((A @ B) @ z, A @ (B @ z))
all_checks.append(ok)
print(f"{PASS if ok else FAIL} (AB)z == A(Bz)")

✓ A(x + y) == Ax + Ay
✓ A(ax) == a(Ax)
✓ A(ax + by) == aAx + bAy
✓ A(0) == 0
✓ (AB)z == A(Bz)


---
## Demo 4: Inner Product → Norm → Metric Hierarchy (Danka §1.03–1.04)

### Theory

Each level adds geometric structure:

| Level | What it gives you | Builds on |
|-------|-------------------|-----------|
| **Inner product** $\langle \cdot, \cdot \rangle$ | Angles, similarity, projection | — |
| **Norm** $\|\cdot\|$ | Magnitude, distance from origin | $\|\mathbf{x}\| = \sqrt{\langle \mathbf{x}, \mathbf{x} \rangle}$ |
| **Metric** $d(\cdot, \cdot)$ | Distance between any two points | $d(\mathbf{x}, \mathbf{y}) = \|\mathbf{x} - \mathbf{y}\|$ |

**Polarisation identity** — recovers the inner product from the norm:

$$\langle \mathbf{x}, \mathbf{y} \rangle = \frac{1}{2}\left(\|\mathbf{x} + \mathbf{y}\|^2 - \|\mathbf{x}\|^2 - \|\mathbf{y}\|^2\right)$$

In [17]:
x = np.array([1.0, 2.0, 3.0])
y = np.array([4.0, -1.0, 2.0])

# Inner product → norm
norm_from_ip = np.sqrt(np.dot(x, x))
norm_direct = np.linalg.norm(x)
ok = np.allclose(norm_from_ip, norm_direct)
all_checks.append(ok)
print(f"√⟨x,x⟩ = {norm_from_ip:.4f},  ||x||₂ = {norm_direct:.4f}")
print(f"{PASS if ok else FAIL} ||x||₂ == √⟨x,x⟩")
print()

# Metric axiom: symmetry
ok = np.allclose(np.linalg.norm(x - y), np.linalg.norm(y - x))
all_checks.append(ok)
print(f"d(x,y) = {np.linalg.norm(x-y):.4f},  d(y,x) = {np.linalg.norm(y-x):.4f}")
print(f"{PASS if ok else FAIL} d(x,y) == d(y,x)")
print()

# Metric axiom: triangle inequality
z = np.zeros(3)
d_xz = np.linalg.norm(x - z)
d_xy = np.linalg.norm(x - y)
d_yz = np.linalg.norm(y - z)
ok = d_xz <= d_xy + d_yz + 1e-10
all_checks.append(ok)
print(f"d(x,z) = {d_xz:.4f}  ≤  d(x,y) + d(y,z) = {d_xy+d_yz:.4f}")
print(f"{PASS if ok else FAIL} Triangle inequality")
print()

# Polarisation identity
ip_direct = np.dot(x, y)
ip_polar = 0.5 * (np.linalg.norm(x+y)**2 - np.linalg.norm(x)**2 - np.linalg.norm(y)**2)
ok = np.allclose(ip_direct, ip_polar)
all_checks.append(ok)
print(f"⟨x,y⟩ direct = {ip_direct:.4f}")
print(f"⟨x,y⟩ polar  = {ip_polar:.4f}")
print(f"{PASS if ok else FAIL} Polarisation identity")

√⟨x,x⟩ = 3.7417,  ||x||₂ = 3.7417
✓ ||x||₂ == √⟨x,x⟩

d(x,y) = 4.3589,  d(y,x) = 4.3589
✓ d(x,y) == d(y,x)

d(x,z) = 3.7417  ≤  d(x,y) + d(y,z) = 8.9415
✓ Triangle inequality

⟨x,y⟩ direct = 8.0000
⟨x,y⟩ polar  = 8.0000
✓ Polarisation identity


---
## Final Report

In [18]:
passed = sum(all_checks)
total = len(all_checks)
print(f"RESULTS: {passed}/{total} checks passed")
assert passed == total, f"{total - passed} checks failed!"
print("OK")

RESULTS: 24/24 checks passed
OK
